# Validation Test Evaluation - Static droplet on slip wall (transient simulation)

This notebook and the corresponding simulation setup notebook (StaticDropletOnSlipWall_transient_Run.ipynb) are part of published results (cf. 5.1.3) found in *The extended discontinuous Galerkin method adapted for moving contact line problems via the generalized Navier boundary condition* (https://doi.org/10.1002/fld.5016).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
wmg.Init("CLpaper_StaticDropletOnSlipWall");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\CLpaper_StaticDropletOnSlipWall"); 
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct29_000000.CLpaper_StaticDropletOnSlipWall");

Opening existing database '\\fdygitrunner\ValidationTests\databases\bkup-2025Oct29_000000.CLpaper_StaticDropletOnSlipWall'.


In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
static var sessions = wmg.Sessions;
//static var sessions = databases.Pick(0).Sessions;
sessions

#0: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k4_mesh0	6/24/2020 11:34:13 PM	f3b1c20e...
#1: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta0_cAstat90_ConvStudy2_k2_mesh2	6/24/2020 11:02:51 PM	f00c7a0d...
#2: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh4	6/24/2020 11:31:44 PM	d806a711...
#3: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta0_cAstat120_ConvStudy2_k2_mesh0	6/24/2020 11:14:55 PM	cc0de00f...
#4: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh0	6/24/2020 11:30:56 PM	ba5e832a...
#5: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta0_cAstat120_ConvStudy2_k4_mesh1	6/24/2020 11:16:53 PM	b189308f...
#6: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta0_cAstat90_ConvStudy2_k4_mesh5	6/24/2020 11:05:18 PM	abb2b814...
#7: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta0_cAstat90_C

## Spurious velocities — Transient simulations

In [5]:
int[] Meshes = { 20, 40, 60 };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct29_000000.CLpaper_StaticDropletOnSlipWall");

In [6]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (int mesh in Meshes) {
    string studyName = $"StaticDropletOnWall_transient_meshStudy_mesh{mesh}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found!");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }

    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
evalSess

Searching for: StaticDropletOnWall_transient_meshStudy_mesh20
found
Searching for: StaticDropletOnWall_transient_meshStudy_mesh40
found
Searching for: StaticDropletOnWall_transient_meshStudy_mesh60
found


#0: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_transient_meshStudy_mesh20	5/10/2021 5:24:44 PM	b1acd49a...
#1: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_transient_meshStudy_mesh40	5/10/2021 5:24:47 PM	40af0fa5...
#2: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_transient_meshStudy_mesh60	5/10/2021 5:24:50 PM	04b5a8ff...


In [8]:
NUnit.Framework.Assert.AreEqual(3, evalSess.Count, $"Not enough sessions for evaluation.");

### compute times

In [10]:
bool calcComputeTimes = false;

In [11]:
if (calcComputeTimes) {

foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}
}

### Evaluate terminal time step

In [12]:
public static void EvaluateTerminalTimeStep(ISessionInfo sess, double refL2velocity, double refL2pressure) {

    string[] nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string meshName;
    int nameLength = nameData.Length;
    if (nameData[nameLength - 1].ToLower().StartsWith("restart"))
        meshName = nameData[nameLength - 2];
    else    
        meshName = nameData.Last();
   

    var terminalStep = sess.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last();

    Console.WriteLine($"{meshName}: time = {terminalStep.PhysicalTime}");

    // L2 error velocity
    double uL2 = terminalStep.GetField("VelocityX").L2Norm();
    double vL2 = terminalStep.GetField("VelocityY").L2Norm();
    double velL2 = (uL2.Pow2()+vL2.Pow2()).Sqrt();
    Console.WriteLine($"L2-error velocity: {velL2}");
    NUnit.Framework.Assert.LessOrEqual(velL2, refL2velocity, 
                $"L2-error for velocity does not match reference value {refL2velocity}.");

    // =======================================
    DGField phi = terminalStep.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi); 
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(125.0);

    // get pressure niveau outside droplet
    DGField pressure = terminalStep.GetField("Pressure");
    int order        = 8;
    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order, 1).XQuadSchemeHelper;
    SpeciesId spcBId = LsTrk.SpeciesIdS[1];
    var vqs = SchemeHelper.GetVolumeQuadScheme(spcBId);
    double areaB     = XNSEUtils.GetSpeciesArea(LsTrk, spcBId);
    double p0        = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat, vqs.Compile(LsTrk.GridDat, order),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            for (int i = 0; i < Length; i++) {
                double p0i = ((XDGField)pressure).GetSpeciesShadowField("B").GetMeanValue(i0+i);
                for (int k = 0; k < QR.NoOfNodes; k++) {
                    EvalResult[i,k,0] = p0i;
                }
            }
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for (int i = 0; i < Length; i++) {
                p0 += ResultsOfIntegration[i, 0]/areaB;
            }
        }
    ).Execute();

    // L2 error outside droplet (spceciesB)
    var pOutN = ((XDGField)pressure).GetSpeciesShadowField("B");
    pOutN.AccConstant(-p0);
    double pOutL2 = pOutN.L2Error((ScalarFunction)null, vqs);

    // L2 error inside droplet (spceciesA)
    SpeciesId spcAId = LsTrk.SpeciesIdS[0];
    vqs   = SchemeHelper.GetVolumeQuadScheme(spcAId);
    double sigma = 1.0;
    double r     = 0.25;
    Func<double[], double> refA = (X => sigma/r);
    var pInN     = ((XDGField)pressure).GetSpeciesShadowField("A");
    pInN.AccConstant(-p0);
    double pInL2 = pInN.L2Error(refA.Vectorize(), vqs);

    // L2 error pressure
    double pL2 = (pInL2.Pow2()+pOutL2.Pow2()).Sqrt();  
    Console.WriteLine($"L2-error pressure: {pL2}");
    NUnit.Framework.Assert.LessOrEqual(pL2, refL2pressure, 
                $"L2-error for pressure does not match reference value {refL2pressure}.");

}

### TABLE 2 - L2-error norms of spurious velocities and Laplace-Young equation for the terminal time step at t = 125

L2-error norms for mesh 20

In [13]:
EvaluateTerminalTimeStep(evalSess.Pick(0), 1.85e-6, 1.24e-3)

mesh20: time = 124.9999999999588
L2-error velocity: 1.848491134491099E-06
L2-error pressure: 0.0011229542726326103


L2-error norms for mesh 40

In [23]:
EvaluateTerminalTimeStep(evalSess.Pick(1), 4.52e-7, 6.53e-4)

mesh40: time = 124.9999999999588
L2-error velocity: 4.5170526232671917E-07
L2-error pressure: 0.0006520934830090356


L2-error norms for mesh 60

In [15]:
//EvaluateTerminalTimeStep(evalSess.Pick(2), 3.21e-7, 5.43e-4)
EvaluateTerminalTimeStep(evalSess.Pick(2), 6.67e-7, 1.40e-3)

mesh60: time = 124.9999999999588
L2-error velocity: 3.2145223237593557E-07
L2-error pressure: 0.0005478025846674979


### Evaluate temporal evolution

In [9]:
public static List<Plot2Ddata> EvaluateData(this List<ISessionInfo> procSess) {
    
    List<Plot2Ddata> plotData = new List<Plot2Ddata>();

    int numberSessions = procSess.Count();

    string[] values = { "spurious velocities", "interface changerate" };
    int numberValues = values.Length;
    //for (int vIdx = 0; vIdx < values.Length; vIdx++) {

        double[][] times = new double[numberSessions][];
        double[][][] valueDatas = new double[numberSessions][][];

        // Read all data
        for (int j = 0; j < numberSessions; j++) {

            var sess = procSess.ElementAt(j);
            Console.WriteLine("Processing session: " + sess.Name);

            // Get session history
            List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
            restartSessionList.Add(sess);
            Guid restartID = sess.RestartedFrom;

            while(restartID != Guid.Empty) {
                try {
                    var rSess = procSess.Where(sess => sess.ID == restartID).Single();
                    Console.WriteLine("  Found restart session: " + rSess.Name);
                    restartSessionList.Add(rSess);
                    restartID = rSess.RestartedFrom;

                } catch { // do nothing if session not found
                    restartID = Guid.Empty;
                }
            }
            restartSessionList.Reverse();

            Console.WriteLine("  Merging data from " + restartSessionList.Count + " sessions.");

            List<double> time = new List<double>();
            List<double>[] valueData = new List<double>[numberValues];
            for(int vIdx = 0; vIdx < numberValues; vIdx++) {
                valueData[vIdx] = new List<double>();
            }

            for(int rSi = 0; rSi < restartSessionList.Count(); rSi++) {

                var timesteps = restartSessionList.ElementAt(rSi).Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
            
                for (int i = 0; i < timesteps.Count(); i++) {

                    var ts = timesteps.Pick(i);
                    double l_time = ts.PhysicalTime;
                    double[] l_values = new double[numberValues];
                
                    DGField phi = ts.GetField("Phi");
                    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
                    LevSet.Acc(1.0, phi);           
                    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
                    LsTrk.UpdateTracker(ts.PhysicalTime);  
         
                    XDGField VelX = (XDGField)ts.GetField("VelocityX");
                    XDGField VelY = (XDGField)ts.GetField("VelocityY");
                    XDGField[] Velocity = new XDGField[] {VelX, VelY};  

                    // get spurious velocities 
                    {
                        double spuriousVelocity = 0.0;                        

                        int momentFittingOrder  = 6;
                        var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), momentFittingOrder, 1).XQuadSchemeHelper;
                                    
                        for(int iSpc = 0; iSpc < LsTrk.SpeciesIdS.Count; iSpc++) {
                            SpeciesId spId = LsTrk.SpeciesIdS[iSpc];

                            var scheme = SchemeHelper.GetVolumeQuadScheme(spId);

                            int D = LsTrk.GridDat.SpatialDimension;
                            for(int d = 0; d < D; d++) {
                                DGField U = Velocity.ElementAt(d);
                                spuriousVelocity += U.L2Error(null, momentFittingOrder, scheme);
            
                            }
                        } 
                        l_values[0] = spuriousVelocity;
                    } 
 
                    // get surface divergence
                    {
                        ConventionalDGField[] meanVelocity = XNSEUtils.GetMeanVelocity(Velocity, LsTrk, 1.0, 1.0);
                        double surfaceDivergence = EnergyUtils.GetSurfaceChangerate(LsTrk, meanVelocity, 6);   
                        l_values[1] = surfaceDivergence;
                    } 

                    if(time.IsNullOrEmpty() || l_time > time.Last()) {
                        time.Add(l_time);
                        for(int vIdx = 0; vIdx < numberValues; vIdx++) {
                            valueData[vIdx].Add(l_values[vIdx]);
                        }
                    }

                }
            }

            Console.WriteLine("... Done");
                            
            times[j] = time.ToArray();
            valueDatas[j] = new double[numberValues][];
            for(int vIdx = 0; vIdx < numberValues; vIdx++) {
                valueDatas[j][vIdx] = valueData[vIdx].ToArray();
            }
        
        }

        // Build DataSets
        Console.WriteLine("Build DataSets");
        for(int vIdx = 0; vIdx < numberValues; vIdx++) {

            KeyValuePair<string, double[][]>[] dataRowsValue = new KeyValuePair<string, double[][]>[numberSessions];
            for(int i = 0; i < numberSessions; i++) {
                string sessName = (procSess.Pick(i).Name).Replace("_", "-");
                dataRowsValue[i] = new KeyValuePair<string, double[][]>(sessName, new double[][] { times[i], valueDatas[i][vIdx] });
            }
            Console.WriteLine("Element at {0}: time vs {1}", vIdx, values[vIdx]);
            Plot2Ddata plt = new Plot2Ddata(dataRowsValue);
            plt.Xlabel = "time";
            plt.Ylabel = values[vIdx];
            plotData.Add(plt);
        }

    return plotData;
}

In [10]:
var evalData = evalSess.EvaluateData();

Processing session: StaticDropletOnWall_transient_meshStudy_mesh20
  Merging data from 1 sessions.
... Done
Processing session: StaticDropletOnWall_transient_meshStudy_mesh40
  Merging data from 1 sessions.
... Done
Processing session: StaticDropletOnWall_transient_meshStudy_mesh60
  Merging data from 1 sessions.
... Done
Build DataSets
Element at 0: time vs spurious velocities
Element at 1: time vs interface changerate


In [11]:
public static Plot2Ddata GetEvalDataPlot(List<Plot2Ddata> data, int dataIndex, string[] meshSizes) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "time";
    plt.Ylabel = data.ElementAt(dataIndex).Ylabel;
    
    var datGroups = data.ElementAt(dataIndex).dataGroups;
    int lcIdx = 1;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (meshSizes.Contains(nameData[3])) {
            plt.AddDataGroup(nameData[3], datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 125.0;
    plt.ShowLegend = true;

    return plt;

} 

### FIGURE 6 

In [12]:
Plot2Ddata[,] Fig6 = new Plot2Ddata[1, 2];

string[] meshStudy = new string[] { "mesh20", "mesh40", "mesh60" };
Fig6[0, 0] = GetEvalDataPlot(evalData, 0, meshStudy);
Fig6[0, 1] = GetEvalDataPlot(evalData, 1, meshStudy);

Fig6.ToGnuplot().PlotSVG(xRes:1300,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 5x10 -6 
 
 
 
 
 1x10 -5 
 
 
 
 
 1.5x10 -5 
 
 
 
 
 2x10 -5 
 
 
 
 
 2.5x10 -5 
 
 
 
 
 3x10 -5 
 
 
 
 
 3.5x10 -5 
 
 
 
 
 4x10 -5 
 
 
 
 
 0 
 
 
 
 
 20 
 
 
 
 
 40 
 
 
 
 
 60 
 
 
 
 
 80 
 
 
 
 
 100 
 
 
 
 
 120 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 spurious velocities 
 
 
 
 
 time 
 
 
 
 
 mesh20 
 
 
 
 
 mesh20 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M454.2,62.1 L507.6,62.1 M111.5,437.5 L111.9,406.2 L112.3,339.1 L112.7,259.4 L113.0,188.4 L113.4,135.1
 L113.8,101.3 L114.2,83.8 L114.6,79.4 L115.0,84.8 L115.4,94.7 L115.7,106.4 L116.1,118.6 L116.5,130.1
 L116.9,140.2 L117.3,148.6 L117.7,154.6 L118.1,160.0 L118.4,164.2 L118.8,167.6 L119.2,170.6 L119.6,173.4
 L120.0,176.4 L120.4,179.7 L120.8,183.6 L121.1,188.2 L121.5,193.6 L121.9,199.8 L122.3,206.7 L122.7,214.2
 L123.1,222.3 L123.5,230.6 L123.8,239.2 L124.2,247.5 L124.6,255.7 L125.0,263.7 L125.4,271.3 L125.8,278.2
 L126.2,284.5 L126.5,289.9 L126.9,294.3 L127.3,297.8 L127.7,300.2 L128.1,301.7 L128.5,302.3 L128.9,302.1
 L129.2,301.3 L129.6,300.0 L130.0,298.3 L130.4,296.5 L130.8,294.6 L131.2,292.7 L131.6,291.0 L131.9,289.5
 L132.3,289.2 L132.7,289.5 L133.1,289.5 L133.5,290.0 L133.9,290.4 L134.3,290.9 L134.6,291.8 L135.0,293.1
 L135.4,294.8 L135.8,296.9 L136.2,299.3 L136.6,302.1 L137.0,305.1 L137.3,308.3 L137.7,311.7 L138.1,315.2
 L138.5,318.8 L138.9,322.3 L139.3,325.8 L139.7,329.1 L140.0,332.1 L140.4,334.9 L140.8,337.3 L141.2,339.3
 L141.6,340.8 L142.0,341.9 L142.4,342.6 L142.7,342.8 L143.1,342.6 L143.5,342.1 L143.9,341.3 L144.3,340.2
 L144.7,339.0 L145.1,337.6 L145.4,336.2 L145.8,334.8 L146.2,333.5 L146.6,332.3 L147.0,331.2 L147.4,330.2
 L147.8,329.5 L148.1,329.0 L148.5,328.7 L148.9,330.9 L149.3,332.1 L149.7,332.8 L150.1,333.7 L150.5,334.7
 L150.8,335.9 L151.2,337.3 L151.6,339.0 L152.0,340.9 L152.4,343.1 L152.8,345.4 L153.2,347.9 L153.5,350.6
 L153.9,353.3 L154.3,356.2 L154.7,359.1 L155.1,362.1 L155.5,365.0 L155.9,368.0 L156.2,370.8 L156.6,373.6
 L157.0,376.3 L157.4,378.8 L157.8,381.2 L158.2,383.4 L158.6,385.4 L158.9,387.1 L159.3,388.6 L159.7,389.9
 L160.1,390.9 L160.5,391.7 L160.9,392.3 L161.3,392.7 L161.6,392.9 L162.0,392.9 L162.4,392.8 L162.8,392.7
 L163.2,392.4 L163.6,392.1 L164.0,391.7 L164.3,391.3 L164.7,390.9 L165.1,390.5 L165.5,390.0 L165.9,389.6
 L166.3,389.1 L166.7,388.7 L167.0,388.2 L167.4,387.7 L167.8,387.3 L168.2,386.7 L168.6,386.2 L169.0,385.7
 L169.4,385.1 L169.7,384.4 L170.1,383.5 L170.5,382.4 L170.9,381.3 L171.3,380.4 L171.7,379.7 L172.1,379.1
 L172.4,378.5 L172.8,377.9 L173.2,377.3 L173.6,376.7 L174.0,376.1 L174.4,375.4 L174.8,374.7 L175.1,374.0
 L175.5,373.4 L175.9,372.7 L176.3,372.1 L176.7,371.5 L177.1,370.9 L177.5,370.3 L177.8,369.8 L178.2,369.3
 L178.6,368.8 L179.0,368.4 L179.4,368.0 L179.8,367.6 L180.2,367.3 L180.5,367.0 L180.9,366.7 L181.3,366.5
 L181.7,366.3 L182.1,366.1 L182.5,365.9 L182.9,365.8 L183.2,365.6 L183.6,365.5 L184.0,365.4 L184.4,365.3
 L184.8,365.1 L185.2,365.0 L185.6,364.9 L185.9,364.8 L186.3,364.6 L186.7,364.5 L187.1,364.3 L187.5,364.1
 L187.9,363.9 L188.3,363.7 L188.6,363.5 L189.0,363.3 L189.4,363.0 L189.8,362.8 L190.2,362.6 L190.6,362.3
 L191.0,362.1 L191.3,361.9 L191.7,361.6 L192.1,361.4 L192.5,361.2 L192.9,361.1 L193.3,360.9 L193.6,360.7
 L194.0,360.6 L194.4,360.5 L194.8,360.4 L195.2,360.4 L195.6,360.4 L196.0,360.4 L196.3,360.4 L196.7,360.4
 L197.1,360.5 L197.5,360.6 L197.9,360.8 L198.3,361.0 L198.7,361.2 L199.0,361.4 L199.4,361.6 L199.8,361.9
 L200.2,362.2 L200.6,362.6 L201.0,362.9 L201.4,363.3 L201.7,363.7 L202.1,364.1 L202.5,364.5 L202.9,365.0
 L203.3,365.5 L203.7,365.9 L204.1,366.4 

In [13]:
public static void EvaluateTerminalTimeStep(Plot2Ddata pltDat, double[] lowerThrshldValues, double[] upperThrshldValues) {

    int numGrps = pltDat.dataGroups.Count();
    if (lowerThrshldValues.Length != numGrps || upperThrshldValues.Length != numGrps) {
        throw new ArgumentException("Threshold value arrays must match number of data groups.");
    }

    for (int i = 0; i < numGrps; i++) {
        var grp = pltDat.dataGroups[i];
        // last time step index
        int timeIdx  = grp.Abscissas.Length - 1;
        double time = grp.Abscissas[timeIdx];

        double valAtTime = grp.Values[timeIdx];
        Console.WriteLine($"Data group: {grp.Name}, value at time {time}: {valAtTime}");

        NUnit.Framework.Assert.IsTrue(valAtTime >= lowerThrshldValues[i], 
            $"Value {valAtTime} of data group {grp.Name} at time {time} is below lower threshold {lowerThrshldValues[i]}.");

        NUnit.Framework.Assert.IsTrue(valAtTime <= upperThrshldValues[i], 
            $"Value {valAtTime} of data group {grp.Name} at time {time} is above upper threshold {upperThrshldValues[i]}.");
    }
}

In [14]:
EvaluateTerminalTimeStep(Fig6[0,0], new double[] { 0.0, 0.0, 0.0 }, new double[] { 5e-6, 5e-6, 5e-6 } );

Data group: mesh20, value at time 124.9999999999588: 3.6830153140714646E-06
Data group: mesh40, value at time 124.9999999999588: 9.14295561880048E-07
Data group: mesh60, value at time 124.9999999999588: 6.456242338347213E-07


In [15]:
EvaluateTerminalTimeStep(Fig6[0,0], new double[] { -5e-6, -5e-6, -5e-6  }, new double[] { 5e-6, 5e-6, 5e-6 } );

Data group: mesh20, value at time 124.9999999999588: 3.6830153140714646E-06
Data group: mesh40, value at time 124.9999999999588: 9.14295561880048E-07
Data group: mesh60, value at time 124.9999999999588: 6.456242338347213E-07
